# 五特征对比：build-3 自定义函数 vs batch compute_all_features

使用同一 TDMS 数据文件、相同滑窗与预处理方式，分别用两种方法计算五个特征并对比差异。

- **方法A**：来自 `2026-05-19-realdata_feature_dataset_build-3.ipynb` 的自定义函数（`_spectral_centroid_mean` 等）
- **方法B**：来自 `2026-05-06-batch_feature_extract_12folders_gpu.ipynb` 的 `compute_all_features` 完整函数


## 1. 模块导入与配置


In [1]:
from __future__ import annotations
from dataclasses import replace
from pathlib import Path
import sys

import numpy as np
import pandas as pd

workspace = Path.cwd()
if not (workspace / "src").exists():
    workspace = workspace.parent
if str(workspace / "src") not in sys.path:
    sys.path.insert(0, str(workspace / "src"))

from fea_cpt_gpu.base import FeatureRecord
from fea_cpt_gpu.params import DEFAULT_FEATURE_PARAMS
from fea_cpt_gpu.signal_ops import build_context, butter_filter
from fea_cpt_gpu.features import compute_all_features

try:
    from nptdms import TdmsFile
except ImportError:
    TdmsFile = None

print("模块加载完成")
print(f"workspace = {workspace}")
print(f"nptdms = {'available' if TdmsFile is not None else 'missing'}")


d:\anaconda3\envs\py312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


模块加载完成
workspace = e:\codes\ZZ-BK
nptdms = available


## 2. 参数与特征配置


In [5]:
# =========================
# 配置
# =========================
DATA_FILE = Path(r"G:\20260323_ZZ_pccp\FIP\24-900-1800\test2\0002341-500K-20260324T201052.395.tdms")

WINDOW_DURATION_S = 0.02       # 20 ms 每窗口
WINDOW_OVERLAP = 0.50         # 50% 重叠

BANDS = {
    "b_1k_100k": (1_000.0, 100_000.0),
    "b_1k_10k":  (1_000.0, 10_000.0),
}

PREPROC_BAND = (1_000.0, 95_000.0)

SELECTED_FEATURES = [
    "b_1k_10k__SC_mean",
    "b_1k_10k__C_f",
    "b_1k_100k__epsilon_2x",
    "b_1k_100k__SC_res_mean",
    "b_1k_100k__I_burst",
]

print(f"数据文件: {DATA_FILE}")
print(f"窗口: {WINDOW_DURATION_S}s, 重叠: {WINDOW_OVERLAP}")
print(f"目标特征: {SELECTED_FEATURES}")


数据文件: G:\20260323_ZZ_pccp\FIP\24-900-1800\test2\0002341-500K-20260324T201052.395.tdms
窗口: 0.02s, 重叠: 0.5
目标特征: ['b_1k_10k__SC_mean', 'b_1k_10k__C_f', 'b_1k_100k__epsilon_2x', 'b_1k_100k__SC_res_mean', 'b_1k_100k__I_burst']


## 3. TDMS 加载辅助函数


In [3]:
# =========================
# TDMS 加载辅助函数（与 build-3 完全一致）
# =========================

def _scalar_text(value):
    if value is None:
        return ""
    if isinstance(value, bytes):
        return value.decode("utf-8", errors="ignore").strip()
    if isinstance(value, np.generic):
        value = value.item()
    if hasattr(value, "tolist") and not isinstance(value, str):
        try:
            value = value.tolist()
        except Exception:
            pass
    if isinstance(value, (list, tuple)) and len(value) == 1:
        return _scalar_text(value[0])
    return str(value).strip()


def _first_property(props, names):
    normalized = {str(k).lower(): v for k, v in props.items()}
    for name in names:
        if name.lower() in normalized:
            return normalized[name.lower()]
    return None


def _coerce_float(value):
    if value is None:
        return None
    try:
        arr = np.asarray(value)
        if arr.shape == ():
            return float(arr.item())
        if arr.size == 1:
            return float(arr.reshape(()).item())
    except Exception:
        pass
    try:
        return float(value)
    except Exception:
        return None


def _select_tdms_channel(tdms_file):
    pref = {"phase_data", "signal", "data", "values", "channel0", "ch0"}
    for g in tdms_file.groups():
        for c in g.channels():
            if str(c.name).lower() in pref:
                return g, c
    best = None
    best_len = -1
    for g in tdms_file.groups():
        for c in g.channels():
            try:
                arr = np.asarray(c[:])
                if arr.size == 0:
                    continue
                if not np.issubdtype(arr.dtype, np.number):
                    arr = arr.astype(float)
            except Exception:
                continue
            if arr.size > best_len:
                best_len = arr.size
                best = (g, c)
    if best is None:
        raise ValueError("No usable numeric channel found in TDMS")
    return best


print("TDMS 辅助函数定义完成")


TDMS 辅助函数定义完成


## 4. 加载 TDMS 数据并预处理


In [9]:
# =========================
# 加载 TDMS 数据并预处理
# =========================

td = TdmsFile.read(DATA_FILE)
g, c = _select_tdms_channel(td)

raw_signal = np.asarray(c[:], dtype=float)

props = {}
props.update(getattr(td, "properties", {}) or {})
props.update(getattr(g, "properties", {}) or {})
props.update(getattr(c, "properties", {}) or {})

sample_rate = _coerce_float(_first_property(props, ("sample_rate", "sample_rate_hz", "sampling_rate", "sampling_rate_hz")))
if sample_rate is None:
    wf_inc = _coerce_float(_first_property(props, ("wf_increment",)))
    if wf_inc and wf_inc > 0:
        sample_rate = 1.0 / wf_inc
sample_rate = sample_rate or 500_000.0  

print(f"采样率: {sample_rate} Hz")
print(f"原始信号长度: {len(raw_signal)} 样本")
print(f"原始信号时长: {len(raw_signal) / sample_rate:.4f} s")

# 预处理：去均值 + 带通滤波（与两个 notebook 完全一致）
centered = raw_signal - float(np.mean(raw_signal))
signal_pre = butter_filter(centered, sample_rate=sample_rate, band_hz=PREPROC_BAND, order=4)

print(f"预处理后信号长度: {len(signal_pre)} 样本")
print(f"预处理后信号时长: {len(signal_pre) / sample_rate:.4f} s")

采样率: 500000.0 Hz
原始信号长度: 5000000 样本
原始信号时长: 10.0000 s
预处理后信号长度: 5000000 样本
预处理后信号时长: 10.0000 s


## 5. 频带参数构建


In [10]:
# =========================
# 构建频带参数（与两个 notebook 完全一致）
# =========================

def _safe_band(low, high, nyq):
    low = max(1.0, min(low, nyq * 0.98))
    high = max(low + 1.0, min(high, nyq * 0.995))
    return (float(low), float(high))


def build_params_for_band(band, sample_rate):
    low, high = band
    nyq = sample_rate / 2.0
    low, high = _safe_band(low, high, nyq)
    span = max(high - low, 10.0)

    low_band = _safe_band(low, low + 0.30 * span, nyq)
    mid_band = _safe_band(low + 0.30 * span, low + 0.60 * span, nyq)
    high1_band = _safe_band(low + 0.50 * span, low + 0.80 * span, nyq)
    high2_band = _safe_band(low + 0.60 * span, high, nyq)
    harmonic_band = _safe_band(low + 0.50 * span, high, nyq)
    ridge_main = _safe_band(low, low + 0.65 * span, nyq)
    ridge_h2 = _safe_band(max(low * 2.0, low + 0.20 * span), min(high * 2.0, nyq * 0.995), nyq)

    return replace(
        DEFAULT_FEATURE_PARAMS,
        highpass_hz=1_000.0,
        main_band_hz=(low, high),
        low_band_hz=low_band,
        mid_band_hz=mid_band,
        high1_band_hz=high1_band,
        high2_band_hz=high2_band,
        harmonic_band_hz=harmonic_band,
        ridge_main_search_hz=ridge_main,
        ridge_h2_search_hz=ridge_h2,
        n_jobs=1,
    )


params_map = {k: build_params_for_band(v, sample_rate) for k, v in BANDS.items()}
print("频带参数构建完成")
for band_name, p in params_map.items():
    print(f"  {band_name}: main_band={p.main_band_hz}, ridge_main={p.ridge_main_search_hz}, ridge_h2={p.ridge_h2_search_hz}")


频带参数构建完成
  b_1k_100k: main_band=(1000.0, 100000.0), ridge_main=(1000.0, 65350.0), ridge_h2=(20800.0, 200000.0)
  b_1k_10k: main_band=(1000.0, 10000.0), ridge_main=(1000.0, 6850.0), ridge_h2=(2800.0, 20000.0)


## 6. 滑窗生成


In [12]:
# =========================
# 生成滑窗
# =========================

def list_window_ranges(n_samples, sample_rate, window_duration_s, overlap):
    win = int(round(window_duration_s * sample_rate))
    if win <= 0:
        raise ValueError("window_samples must be positive")
    if n_samples < win:
        return []
    step = max(1, int(round(win * (1.0 - overlap))))
    out = []
    idx = 0
    wid = 0
    while idx + win <= n_samples:
        out.append((wid, idx, idx + win, win, step))
        idx += step
        wid += 1
    return out


n_samples = len(signal_pre)
windows = list_window_ranges(n_samples, sample_rate, WINDOW_DURATION_S, WINDOW_OVERLAP)
print(f"总窗口数: {len(windows)}")
if windows:
    print(f"窗口大小: {windows[0][3]} 样本 ({WINDOW_DURATION_S*1000:.0f} ms)")
    print(f"窗口步长: {windows[0][4]} 样本")


总窗口数: 999
窗口大小: 10000 样本 (20 ms)
窗口步长: 5000 样本


## 7. 方法A：build-3 自定义特征函数


In [13]:
# =========================
# 方法A：build-3 notebook 的自定义特征函数
# =========================

def _spectral_centroid_mean_A(freqs, power, eps):
    """与 build-3 中 _spectral_centroid_mean 完全一致"""
    if power.size == 0:
        return 0.0
    numerator = np.sum(freqs[:, None] * power, axis=0)
    denominator = np.sum(power, axis=0) + eps
    sc = numerator / denominator
    return float(np.mean(sc)) if sc.size else 0.0


def _c_f_from_context_A(context):
    """与 build-3 中 _c_f_from_context 完全一致"""
    eps = context.params.eps
    if len(context.ridge_f1) <= 2:
        return 0.0
    curvature = np.gradient(
        np.gradient(context.ridge_f1, context.stft_times + eps),
        context.stft_times + eps,
    )
    return float(np.mean(np.abs(curvature) / (np.mean(np.abs(context.ridge_f1)) + eps))) if curvature.size else 0.0


def _epsilon_2x_from_context_A(context):
    """与 build-3 中 _epsilon_2x_from_context 完全一致"""
    eps = context.params.eps
    active = context.ridge_f1 > 0.0
    if not np.any(active):
        return 0.0
    diff_h2 = np.abs(context.ridge_f2 - 2.0 * context.ridge_f1)
    return float(np.median(diff_h2[active] / (context.ridge_f1[active] + eps)))


def _sc_res_mean_from_context_A(context):
    """与 build-3 中 _sc_res_mean_from_context 完全一致"""
    eps = context.params.eps
    sc_res = _spectral_centroid_mean_A(context.stft_freqs, context.residual_power, eps)
    return float(sc_res)


def _i_burst_from_context_A(context):
    """与 build-3 中 _i_burst_from_context 完全一致"""
    eps = context.params.eps
    total_wp = float(sum(context.wavelet_node_energies.values())) + eps
    sorted_nodes = sorted(context.wavelet_node_energies.items())
    wp_prob = np.asarray([energy / total_wp for _, energy in sorted_nodes], dtype=float)
    return float(np.max(wp_prob) / (np.median(wp_prob) + eps)) if wp_prob.size else 0.0


def compute_features_method_A(window_signal, sample_rate, params_map):
    """方法A：使用 build-3 notebook 的自定义函数提取5个特征"""
    rec = FeatureRecord(
        sample_id="w", sample_name="w", sample_type="raw", sample_type_code=0,
        path=Path("."), signal=np.asarray(window_signal, dtype=float),
        sample_rate=float(sample_rate), metadata={},
    )
    out = {}

    ctx_1k10k = build_context(rec, params_map["b_1k_10k"])
    out["b_1k_10k__SC_mean"] = _spectral_centroid_mean_A(ctx_1k10k.stft_freqs, ctx_1k10k.stft_power, ctx_1k10k.params.eps)
    out["b_1k_10k__C_f"] = _c_f_from_context_A(ctx_1k10k)

    ctx_1k100k = build_context(rec, params_map["b_1k_100k"])
    out["b_1k_100k__epsilon_2x"] = _epsilon_2x_from_context_A(ctx_1k100k)
    out["b_1k_100k__SC_res_mean"] = _sc_res_mean_from_context_A(ctx_1k100k)
    out["b_1k_100k__I_burst"] = _i_burst_from_context_A(ctx_1k100k)

    return out


print("方法A（build-3 自定义函数）定义完成")


方法A（build-3 自定义函数）定义完成


## 8. 方法B：batch compute_all_features


In [14]:
# =========================
# 方法B：batch notebook 的 compute_all_features
# =========================

def compute_features_method_B(window_signal, sample_rate, params_map):
    """方法B：使用 batch notebook 的 compute_all_features 提取5个特征"""
    rec = FeatureRecord(
        sample_id="w", sample_name="w", sample_type="raw", sample_type_code=0,
        path=Path("."), signal=np.asarray(window_signal, dtype=float),
        sample_rate=float(sample_rate), metadata={},
    )
    out = {}

    ctx_1k10k = build_context(rec, params_map["b_1k_10k"])
    result_1k10k = compute_all_features(ctx_1k10k)
    out["b_1k_10k__SC_mean"] = float(result_1k10k.features["SC_mean"])
    out["b_1k_10k__C_f"] = float(result_1k10k.features["C_f"])

    ctx_1k100k = build_context(rec, params_map["b_1k_100k"])
    result_1k100k = compute_all_features(ctx_1k100k)
    out["b_1k_100k__epsilon_2x"] = float(result_1k100k.features["epsilon_2x"])
    out["b_1k_100k__SC_res_mean"] = float(result_1k100k.features["SC_res_mean"])
    out["b_1k_100k__I_burst"] = float(result_1k100k.features["I_burst"])

    return out


print("方法B（batch compute_all_features）定义完成")


方法B（batch compute_all_features）定义完成


## 9. 逐窗口计算两种方法的特征


In [15]:
# =========================
# 逐窗口计算两种方法的特征
# =========================

results_A = []
results_B = []

for wid, i0, i1, win_len, step_len in windows:
    win_signal = signal_pre[i0:i1]

    feat_A = compute_features_method_A(win_signal, sample_rate, params_map)
    feat_B = compute_features_method_B(win_signal, sample_rate, params_map)

    feat_A["window_id"] = wid
    feat_A["window_start_s"] = i0 / sample_rate
    feat_B["window_id"] = wid
    feat_B["window_start_s"] = i0 / sample_rate

    results_A.append(feat_A)
    results_B.append(feat_B)

df_A = pd.DataFrame(results_A)
df_B = pd.DataFrame(results_B)

print(f"方法A 计算完成: {len(df_A)} 窗口")
print(f"方法B 计算完成: {len(df_B)} 窗口")


方法A 计算完成: 999 窗口
方法B 计算完成: 999 窗口


## 10. 对比两种方法的结果


In [16]:
# =========================
# 对比两种方法的结果
# =========================

feature_cols = [c for c in SELECTED_FEATURES if c in df_A.columns and c in df_B.columns]

diff_records = []
for feat_name in feature_cols:
    vals_A = df_A[feat_name].values
    vals_B = df_B[feat_name].values
    abs_diff = np.abs(vals_A - vals_B)
    rel_diff = np.where(
        np.abs(vals_B) > 1e-12,
        abs_diff / np.abs(vals_B),
        abs_diff,
    )
    max_abs = float(np.max(abs_diff))
    max_rel = float(np.max(rel_diff))
    mean_abs = float(np.mean(abs_diff))
    mean_rel = float(np.mean(rel_diff))
    n_exact = int(np.sum(abs_diff == 0))
    n_total = len(abs_diff)
    diff_records.append({
        "特征名": feat_name,
        "最大绝对差": max_abs,
        "最大相对差": max_rel,
        "平均绝对差": mean_abs,
        "平均相对差": mean_rel,
        "完全相同窗口数": n_exact,
        "总窗口数": n_total,
        "是否完全一致": "是" if n_exact == n_total else "否",
    })

df_diff = pd.DataFrame(diff_records)
print("=" * 80)
print("方法A vs 方法B 对比结果")
print("=" * 80)
print(df_diff.to_string(index=False))
print()
all_identical = all(r["是否完全一致"] == "是" for r in diff_records)
if all_identical:
    print("结论：所有五个特征在所有窗口上完全一致！")
else:
    print("结论：存在差异，两种方法结果不完全一致。")


方法A vs 方法B 对比结果
                   特征名  最大绝对差  最大相对差  平均绝对差  平均相对差  完全相同窗口数  总窗口数 是否完全一致
     b_1k_10k__SC_mean    0.0    0.0    0.0    0.0      999   999      是
         b_1k_10k__C_f    0.0    0.0    0.0    0.0      999   999      是
 b_1k_100k__epsilon_2x    0.0    0.0    0.0    0.0      999   999      是
b_1k_100k__SC_res_mean    0.0    0.0    0.0    0.0      999   999      是
    b_1k_100k__I_burst    0.0    0.0    0.0    0.0      999   999      是

结论：所有五个特征在所有窗口上完全一致！


C:\Users\QiGh\AppData\Local\Temp\ipykernel_25892\3767755043.py:14: RuntimeWarning: invalid value encountered in divide
  abs_diff / np.abs(vals_B),


## 11. 详细逐窗口差值


In [17]:
# =========================
# 详细逐窗口差值
# =========================

for feat_name in feature_cols:
    vals_A = df_A[feat_name].values
    vals_B = df_B[feat_name].values
    abs_diff = np.abs(vals_A - vals_B)
    if np.max(abs_diff) > 0:
        print(f"\n--- {feat_name} ---")
        print(f"  方法A 前5个值: {vals_A[:5]}")
        print(f"  方法B 前5个值: {vals_B[:5]}")
        print(f"  绝对差 前5个:  {abs_diff[:5]}")
        worst_idx = int(np.argmax(abs_diff))
        print(f"  最大差值窗口: window_id={worst_idx}, A={vals_A[worst_idx]:.10f}, B={vals_B[worst_idx]:.10f}, diff={abs_diff[worst_idx]:.2e}")
    else:
        print(f"\n--- {feat_name} --- 完全一致")



--- b_1k_10k__SC_mean --- 完全一致

--- b_1k_10k__C_f --- 完全一致

--- b_1k_100k__epsilon_2x --- 完全一致

--- b_1k_100k__SC_res_mean --- 完全一致

--- b_1k_100k__I_burst --- 完全一致


## 12. 最终汇总


In [18]:
# =========================
# 最终汇总
# =========================

print("=" * 80)
print("对比汇总")
print("=" * 80)
print(f"数据文件: {DATA_FILE.name}")
print(f"采样率: {sample_rate} Hz")
print(f"总窗口数: {len(windows)}")
print(f"窗口参数: {WINDOW_DURATION_S}s 窗口, {WINDOW_OVERLAP*100:.0f}% 重叠")
print(f"预处理: 去均值 + {PREPROC_BAND} Hz 带通滤波 (4阶 Butterworth)")
print()
print("方法A: build-3 notebook 自定义函数")
print("方法B: batch notebook compute_all_features")
print()
for r in diff_records:
    status = "一致" if r["是否完全一致"] == "是" else "有差异"
    print(f"  {r['特征名']}: {status} (最大绝对差={r['最大绝对差']:.2e}, 最大相对差={r['最大相对差']:.2e})")
print()
if all_identical:
    print("结论: 两种方法对所有窗口的五个特征计算结果完全一致。")
else:
    print("结论: 两种方法存在数值差异，请查看上方详细差值。")


对比汇总
数据文件: 0002341-500K-20260324T201052.395.tdms
采样率: 500000.0 Hz
总窗口数: 999
窗口参数: 0.02s 窗口, 50% 重叠
预处理: 去均值 + (1000.0, 95000.0) Hz 带通滤波 (4阶 Butterworth)

方法A: build-3 notebook 自定义函数
方法B: batch notebook compute_all_features

  b_1k_10k__SC_mean: 一致 (最大绝对差=0.00e+00, 最大相对差=0.00e+00)
  b_1k_10k__C_f: 一致 (最大绝对差=0.00e+00, 最大相对差=0.00e+00)
  b_1k_100k__epsilon_2x: 一致 (最大绝对差=0.00e+00, 最大相对差=0.00e+00)
  b_1k_100k__SC_res_mean: 一致 (最大绝对差=0.00e+00, 最大相对差=0.00e+00)
  b_1k_100k__I_burst: 一致 (最大绝对差=0.00e+00, 最大相对差=0.00e+00)

结论: 两种方法对所有窗口的五个特征计算结果完全一致。


## 13. 特征概率分布对比：方法A（无标签） vs 方法B（按标签分组）

对比两种计算方式产出的五种特征的概率分布：
- **方法A**：`realdata_feature_dataset_20260519_v3` + `realdata_feature_dataset_20260523` 中所有样本合并，无标签区分
- **方法B**：`dataset_build_260518_features_gpu` 中按 label 列分组（0=噪声，1=断丝），分别绘制分布

每个特征一个子图，横轴为特征值，纵轴为概率密度（KDE）。


In [ ]:
# =========================
# 13.1 读取方法A的特征数据
# =========================
import glob
from scipy.stats import gaussian_kde
import matplotlib.pyplot as plt
import matplotlib

matplotlib.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False

FEATURE_COLS = [
    'b_1k_10k__SC_mean',
    'b_1k_10k__C_f',
    'b_1k_100k__epsilon_2x',
    'b_1k_100k__SC_res_mean',
    'b_1k_100k__I_burst',
]

# 方法A：读取 v3 和 0523 目录下的特征文件
dir_A_v3 = workspace / 'outputs' / 'realdata_feature_dataset_20260519_v3'
dir_A_0523 = workspace / 'outputs' / 'realdata_feature_dataset_20260523'

dfs_A = []
for d in [dir_A_v3, dir_A_0523]:
    pattern = str(d / '*_features_*_part_*.csv')
    for fp in sorted(glob.glob(pattern)):
        dfs_A.append(pd.read_csv(fp, usecols=FEATURE_COLS, encoding='utf-8'))

df_A_all = pd.concat(dfs_A, ignore_index=True)
print(f'方法A 样本总数: {len(df_A_all)}')
print(f'方法A 特征列: {list(df_A_all.columns)}')


In [ ]:
# =========================
# 13.2 读取方法B的特征数据并按标签分组
# =========================

dir_B = workspace / 'outputs' / 'dataset_build_260518_features_gpu'
cols_B = FEATURE_COLS + ['label']

dfs_B = []
for fp in sorted(glob.glob(str(dir_B / '*_features.csv'))):
    dfs_B.append(pd.read_csv(fp, usecols=cols_B, encoding='utf-8'))

df_B_all = pd.concat(dfs_B, ignore_index=True)
df_B_noise = df_B_all[df_B_all['label'] == 0].reset_index(drop=True)
df_B_break = df_B_all[df_B_all['label'] == 1].reset_index(drop=True)

print(f'方法B 样本总数: {len(df_B_all)}')
print(f'  噪声(label=0): {len(df_B_noise)}')
print(f'  断丝(label=1): {len(df_B_break)}')


In [ ]:
# =========================
# 13.3 绘制五种特征概率分布对比图
# =========================

fig, axes = plt.subplots(1, 5, figsize=(28, 5))

for idx, feat in enumerate(FEATURE_COLS):
    ax = axes[idx]

    # 方法A：所有样本 KDE
    vals_A = df_A_all[feat].dropna().values
    if vals_A.size > 1:
        kde_A = gaussian_kde(vals_A)
        x_min = vals_A.min()
        x_max = vals_A.max()
        x_range = x_max - x_min
        if x_range < 1e-12:
            x_range = 1.0
        xs_A = np.linspace(x_min - 0.05 * x_range, x_max + 0.05 * x_range, 500)
        ax.plot(xs_A, kde_A(xs_A), color='#1f77b4', linewidth=2, label='方法A (全样本)')

    # 方法B-噪声
    vals_B0 = df_B_noise[feat].dropna().values
    if vals_B0.size > 1:
        kde_B0 = gaussian_kde(vals_B0)
        x_min0 = vals_B0.min()
        x_max0 = vals_B0.max()
        x_range0 = x_max0 - x_min0
        if x_range0 < 1e-12:
            x_range0 = 1.0
        xs_B0 = np.linspace(x_min0 - 0.05 * x_range0, x_max0 + 0.05 * x_range0, 500)
        ax.plot(xs_B0, kde_B0(xs_B0), color='#ff7f0e', linewidth=2, linestyle='--', label='方法B-噪声')

    # 方法B-断丝
    vals_B1 = df_B_break[feat].dropna().values
    if vals_B1.size > 1:
        kde_B1 = gaussian_kde(vals_B1)
        x_min1 = vals_B1.min()
        x_max1 = vals_B1.max()
        x_range1 = x_max1 - x_min1
        if x_range1 < 1e-12:
            x_range1 = 1.0
        xs_B1 = np.linspace(x_min1 - 0.05 * x_range1, x_max1 + 0.05 * x_range1, 500)
        ax.plot(xs_B1, kde_B1(xs_B1), color='#d62728', linewidth=2, linestyle=':', label='方法B-断丝')

    ax.set_xlabel(feat, fontsize=10)
    ax.set_ylabel('概率密度', fontsize=10)
    ax.legend(fontsize=8, loc='best')
    ax.tick_params(labelsize=8)

fig.suptitle('五种特征概率分布对比：方法A(无标签) vs 方法B(按标签分组)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(workspace / 'outputs' / 'feature_distribution_comparison_A_vs_B.png', dpi=150, bbox_inches='tight')
plt.show()
print('分布对比图已保存至 outputs/feature_distribution_comparison_A_vs_B.png')
